# 01 — OMI quotations

**Purpose:** turn the 44 semiannual OMI releases into a transparent analytical dataset for the residential market.

**Flow:** inventory → consolidation → validation → quotation features → residential analysis.

**Interpretation:** OMI publishes minimum/maximum market quotations in €/m². `Compr_mid` is their arithmetic midpoint; it is not an observed sale price and not a statistical median.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start=None):
    start=Path(start or Path.cwd()).resolve()
    for candidate in [start,*start.parents]:
        if (candidate/'data'/'raw'/'quotations').is_dir(): return candidate
    raise FileNotFoundError('Could not locate data/raw/quotations.')

PROJECT_ROOT=find_project_root()
RAW_DIR=PROJECT_ROOT/'data'/'raw'/'quotations'
FILE_RE=re.compile(r'^omi_quotations_(\d{4})_(S[12])\.csv$',re.I)
records=[]
for path in sorted(RAW_DIR.glob('omi_quotations_*.csv')):
    m=FILE_RE.match(path.name)
    if not m: raise ValueError(f'Unexpected filename: {path.name}')
    records.append({'path':path,'year':int(m.group(1)),'semester':m.group(2).upper()})
catalogue=pd.DataFrame(records).sort_values(['year','semester']).reset_index(drop=True)
if catalogue.empty: raise FileNotFoundError(f'No files in {RAW_DIR}')

frames=[]
for row in catalogue.itertuples(index=False):
    part=pd.read_csv(row.path,sep=';',low_memory=False)
    part['reference_year']=row.year
    part['semester']=row.semester
    part['reference_date']=pd.Timestamp(row.year,6 if row.semester=='S1' else 12,30 if row.semester=='S1' else 31)
    frames.append(part)
omi=pd.concat(frames,ignore_index=True)

print(f'Files: {len(catalogue):,} | Rows: {len(omi):,} | Coverage: {omi.reference_year.min()}–{omi.reference_year.max()}')
display(catalogue[['year','semester']])

## Data-quality and transformation

We validate the expected OMI fields before deriving any metric. Numeric fields are converted explicitly because some releases use comma decimals. Municipality identifiers are kept as strings because they are keys, not quantities.

In [ ]:
required={'Comune_ISTAT','Comune_cat','Comune_amm','Comune_descrizione','Zona','Descr_Tipologia','Compr_min','Compr_max'}
missing=sorted(required-set(omi.columns))
if missing: raise ValueError(f'Missing expected columns: {missing}')

periods=set(map(tuple,omi[['reference_year','semester']].drop_duplicates().to_numpy()))
expected=set(map(tuple,catalogue[['year','semester']].to_numpy()))
print('Missing periods:',sorted(expected-periods))

for col in ['Compr_min','Compr_max','Loc_min','Loc_max']:
    if col in omi.columns:
        omi[col]=pd.to_numeric(omi[col].astype('string').str.replace(',','.',regex=False),errors='coerce')

if (omi['Compr_min']>omi['Compr_max']).any():
    raise ValueError('Found OMI rows where Compr_min > Compr_max.')

positive_mid=(omi['Compr_min']+omi['Compr_max'])/2
nonzero=positive_mid.replace(0,np.nan)
omi['Compr_mid']=positive_mid
omi['Compr_spread']=omi['Compr_max']-omi['Compr_min']
omi['Compr_spread_pct']=omi['Compr_spread'].div(nonzero).mul(100)

raw_code=pd.to_numeric(omi['Comune_ISTAT'],errors='coerce').astype('Int64').astype('string')
omi['municipality_code_omi']=raw_code
omi['municipality_code']=raw_code.str[-6:]

print(f'Missing Compr_mid: {omi.Compr_mid.isna().mean():.2%}')
display(omi[['Compr_min','Compr_max','Compr_mid','Compr_spread','Compr_spread_pct']].describe().T)

## Residential universe

The analysis retains typologies whose description contains `abitazion` or `villa`. This is deliberately visible in the notebook so that the definition of the residential market can be audited. The resulting data remain zone-level: a municipality can have many OMI observations in one semester.

In [ ]:
residential=omi.loc[omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa',case=False,na=False) & omi['Compr_mid'].notna()].copy()

print(f'Residential observations: {len(residential):,}')
print(f'Municipalities: {residential.municipality_code.nunique():,}')
print(f'Semesters: {residential.reference_date.nunique():,}')
display(residential['Descr_Tipologia'].value_counts().rename('observations').to_frame())

national=(residential.groupby('reference_date',as_index=False).agg(median_omi_m2=('Compr_mid','median'),mean_omi_m2=('Compr_mid','mean'),observations=('Compr_mid','count'),municipalities=('municipality_code','nunique')).sort_values('reference_date'))
display(national.tail(10))

fig,ax=plt.subplots(figsize=(11,5))
ax.plot(national.reference_date,national.median_omi_m2)
ax.set(title='Residential OMI quotation midpoint — national median',xlabel='Semester',ylabel='€/m²')
ax.grid(alpha=.25)
plt.show()

latest=residential.reference_date.max()
regional=(residential.loc[residential.reference_date.eq(latest)].groupby('Regione',as_index=False).agg(median_omi_m2=('Compr_mid','median'),observations=('Compr_mid','count'),municipalities=('municipality_code','nunique')).sort_values('median_omi_m2',ascending=False))
display(regional)

## Interpretation and hand-off

This notebook establishes a clean quotation layer. The national statistic is the median of zone-level OMI midpoints, so it is **not transaction-weighted**. The next notebook introduces NTN, which measures market activity rather than prices. Keeping the two concepts separate until integration prevents misleading comparisons.